In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# Load data
df = pd.read_csv("/content/drive/MyDrive/Final_Augmented_dataset_Diseases_and_Symptoms.csv")
symptoms = df.columns[1:]

X = df[symptoms].values
y = df['diseases'].values

print(f"Dataset shape: {df.shape}")
print(f"X shape: {X.shape}")
print(f"Number of symptoms: {len(symptoms)}")

# Check class distribution
class_counts = df['diseases'].value_counts()
print(f"\nClass distribution:")
print(class_counts.head(10))  # Show top 10

# Check for classes with only 1 sample
single_sample_classes = class_counts[class_counts == 1].index.tolist()
print(f"\nClasses with only 1 sample: {len(single_sample_classes)}")

# Remove classes with only 1 sample
if single_sample_classes:
    df = df[~df['diseases'].isin(single_sample_classes)]
    print(f"Removed {len(single_sample_classes)} classes with single samples")
    print(f"New dataset shape: {df.shape}")

    # Update X and y
    X = df[symptoms].values
    y = df['diseases'].values

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

print(f"\nNumber of disease classes: {num_classes}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")

# Calculate proper reshaping
total_samples = X.shape[0]
num_features = X.shape[1]

# Choose timesteps that evenly divides the features
timesteps = 5
# Make sure features per step is integer
features_per_step = num_features // timesteps
# Use only the features that fit evenly
usable_features = features_per_step * timesteps

print(f"\nReshaping parameters:")
print(f"Timesteps: {timesteps}")
print(f"Features per step: {features_per_step}")
print(f"Usable features: {usable_features} out of {num_features}")

# Use only the first usable_features columns
X_trimmed = X[:, :usable_features]

# Reshape for LSTM
X_reshaped = X_trimmed.reshape(total_samples, timesteps, features_per_step)
print(f"Reshaped X shape: {X_reshaped.shape}")

# Scale features - PROPERLY
# Create separate scalers for each timestep
scalers = []
X_scaled = np.zeros_like(X_reshaped, dtype=np.float32)

for i in range(timesteps):
    scaler = MinMaxScaler()
    X_scaled[:, i, :] = scaler.fit_transform(X_reshaped[:, i, :])
    scalers.append(scaler)

# Split data WITHOUT stratification if any class still has too few samples
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_encoded,
        test_size=0.2,
        random_state=42,
        stratify=y_encoded
    )
except ValueError as e:
    print(f"\nStratified split failed: {e}")
    print("Using non-stratified split instead...")
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y_encoded,
        test_size=0.2,
        random_state=42
    )

print(f"\nTraining shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")

# Check train/test class distribution
unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_test, counts_test = np.unique(y_test, return_counts=True)
print(f"\nTrain classes: {len(unique_train)}, Test classes: {len(unique_test)}")

# Build LSTM model
model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True,
                      kernel_regularizer=l2(0.001),
                      recurrent_regularizer=l2(0.001)),
                  input_shape=(X_train.shape[1], X_train.shape[2])),
    BatchNormalization(),
    Dropout(0.4),

    Bidirectional(LSTM(64, return_sequences=False,
                      kernel_regularizer=l2(0.001),
                      recurrent_regularizer=l2(0.001))),
    BatchNormalization(),
    Dropout(0.4),

    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(num_classes, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    min_delta=0.001
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001
)

# Train model
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr],
    verbose=1,
    shuffle=True
)



Dataset shape: (246945, 378)
X shape: (246945, 377)
Number of symptoms: 377

Class distribution:
diseases
cystitis                          1219
vulvodynia                        1218
nose disorder                     1218
complex regional pain syndrome    1217
spondylosis                       1216
esophagitis                       1215
conjunctivitis due to allergy     1215
hypoglycemia                      1215
peripheral nerve disorder         1215
vaginal cyst                      1215
Name: count, dtype: int64

Classes with only 1 sample: 19
Removed 19 classes with single samples
New dataset shape: (246926, 378)

Number of disease classes: 754
Number of samples: 246926
Number of features: 377

Reshaping parameters:
Timesteps: 5
Features per step: 75
Usable features: 375 out of 377
Reshaped X shape: (246926, 5, 75)

Training shape: (197540, 5, 75)
Testing shape: (49386, 5, 75)

Train classes: 754, Test classes: 740


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
6174/6174 ━━━━━━━━━━━━━━━━━━━━ 106s 16ms/step - accuracy: 0.3039 - loss: 3.9878 - val_accuracy: 0.7541 - val_loss: 1.2486 - learning_rate: 0.0010
Epoch 2/5
6174/6174 ━━━━━━━━━━━━━━━━━━━━ 141s 16ms/step - accuracy: 0.5621 - loss: 2.0164 - val_accuracy: 0.7763 - val_loss: 1.1214 - learning_rate: 0.0010
Epoch 3/5
6174/6174 ━━━━━━━━━━━━━━━━━━━━ 100s 16ms/step - accuracy: 0.5962 - loss: 1.8452 - val_accuracy: 0.7818 - val_loss: 1.0788 - learning_rate: 0.0010
Epoch 4/5
6174/6174 ━━━━━━━━━━━━━━━━━━━━ 98s 16ms/step - accuracy: 0.6118 - loss: 1.7666 - val_accuracy: 0.7918 - val_loss: 1.0390 - learning_rate: 0.0010
Epoch 5/5
6174/6174 ━━━━━━━━━━━━━━━━━━━━ 100s 16ms/step - accuracy: 0.6175 - loss: 1.7334 - val_accuracy: 0.7900 - val_loss: 1.0241 - learning_rate: 0.0010


In [ ]:
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f'Training Accuracy: {train_acc*100:.2f}%')
print(f'Testing Accuracy:  {test_acc*100:.2f}%')
print(f'Training Loss:     {train_loss:.4f}')
print(f'Testing Loss:      {test_loss:.4f}')


Training Accuracy: 79.06%
Testing Accuracy:  79.00%
Training Loss:     1.0165
Testing Loss:      1.0241
